# Firefly Algorithm (FA) — Assignment วิชา Optimization

**เงื่อนไข:** เขียนอัลกอริทึมเองทั้งหมด ไม่ใช้ Optimization Library ใดๆ — ใช้เฉพาะ `numpy`, `matplotlib`, `time` และ library มาตรฐานของ Python (`os`)

| ส่วน | เนื้อหา | ฟังก์ชันหลัก |
|---|---|---|
| 0 | พารามิเตอร์ (ปรับได้ที่เดียว) | — |
| 1 | ฟังก์ชันทดสอบ + ตัวนับ NFE | `rosenbrock()`, `rastrigin()`, `CountedFunction` |
| 2 | ตัวอัลกอริทึม FA | `firefly_algorithm()` |
| 3 | โหมดตรวจการคำนวณด้วยมือ | `hand_calculation_demo()` |
| 4 | การทดลอง 30 runs + ตารางสรุป | `run_experiments()`, `print_summary_table()` |
| 5 | กราฟ (บันทึกเป็น .png) | `plot_convergence()`, `plot_swarm_snapshots()` |
| 6 | ปัญหาจริง: Tension/Compression Spring Design | `real_world_demo()` |
| 7 | อธิบายโค้ดเทียบกับสมการ (สำหรับนำเสนอ) | — |

> รันด้วย **Run All** จากบนลงล่าง — กราฟทุกภาพจะถูกบันทึกไว้ในโฟลเดอร์ `fa_figures/` ข้างไฟล์ notebook

## 0. พารามิเตอร์ทั้งหมด (แก้ตรงนี้ที่เดียว)

In [ ]:
import os
import time

import numpy as np
import matplotlib.pyplot as plt

# ================= พารามิเตอร์ของ Firefly Algorithm =================
N_FIREFLIES = 25      # n  : จำนวนหิ่งห้อยในฝูง
MAX_ITER = 100        # T  : จำนวนรอบสูงสุด (ต้องไม่น้อยกว่า 20)
BETA0 = 1.0           # β0 : ความดึงดูดเมื่อระยะห่าง r = 0
GAMMA = 1.0           # γ  : สัมประสิทธิ์การดูดกลืนแสง (ยิ่งมาก ยิ่งมองเห็นกันแค่ระยะใกล้)
ALPHA0 = 0.2          # α0 : ขนาดของการสุ่มในรอบแรก
THETA = 0.97          # θ  : อัตราการลด α ในแต่ละรอบ  α_t = α0 · θ^t

# ================= การตั้งค่าการทดลอง =================
N_RUNS = 30           # จำนวนการรันอิสระต่อฟังก์ชัน
SEED = 42             # seed หลัก — run ที่ k ใช้ seed = SEED + k (ทำซ้ำผลได้)
SUCCESS_TOL = 1e-4    # นับว่า "สำเร็จ" เมื่อได้ f < ค่านี้
SNAPSHOT_ITERS = [0, 5, 20, MAX_ITER]   # รอบที่จะวาดตำแหน่งหิ่งห้อยบน contour
FIG_DIR = "fa_figures"                  # โฟลเดอร์เก็บไฟล์กราฟ .png

assert MAX_ITER >= 20, "MAX_ITER ต้องไม่น้อยกว่า 20"
os.makedirs(FIG_DIR, exist_ok=True)
print(f"n={N_FIREFLIES}, max_iter={MAX_ITER}, beta0={BETA0}, gamma={GAMMA}, "
      f"alpha0={ALPHA0}, theta={THETA}, n_runs={N_RUNS}, seed={SEED}")
print("บันทึกกราฟไว้ที่:", os.path.abspath(FIG_DIR))

## 1. ฟังก์ชันทดสอบ (2 มิติ, หาค่าต่ำสุด)

| ฟังก์ชัน | สูตร | ขอบเขต | ค่าต่ำสุดจริง | ลักษณะ |
|---|---|---|---|---|
| **Rosenbrock** | $f(x,y) = (1-x)^2 + 100(y - x^2)^2$ | $[-5, 5]^2$ | $f = 0$ ที่ $(1, 1)$ | หุบเขารูปกล้วยที่แคบและแบน — หาหุบง่าย แต่เดินไปถึงก้นยาก |
| **Rastrigin** | $f(x,y) = 20 + x^2 + y^2 - 10[\cos 2\pi x + \cos 2\pi y]$ | $[-5.12, 5.12]^2$ | $f = 0$ ที่ $(0, 0)$ | มีหลุมหลอก (local minima) เรียงเป็นตารางราว 100 หลุม |

**NFE (Number of Function Evaluations)** = จำนวนครั้งที่เรียก objective function — เป็นตัววัด "ต้นทุน" ของอัลกอริทึมที่ไม่ขึ้นกับความเร็วเครื่อง
เราห่อฟังก์ชันด้วยคลาส `CountedFunction` ให้บวกตัวนับทุกครั้งที่ถูกเรียก

In [ ]:
def rosenbrock(p):
    # f(x, y) = (1 - x)^2 + 100 (y - x^2)^2      ค่าต่ำสุด f = 0 ที่ (1, 1)
    x, y = p
    return (1 - x) ** 2 + 100 * (y - x ** 2) ** 2


def rastrigin(p):
    # f(x, y) = 20 + x^2 + y^2 - 10[cos(2πx) + cos(2πy)]      ค่าต่ำสุด f = 0 ที่ (0, 0)
    x, y = p
    return 20 + x ** 2 + y ** 2 - 10 * (np.cos(2 * np.pi * x) + np.cos(2 * np.pi * y))


# ข้อมูลของแต่ละฟังก์ชันทดสอบ: ขอบเขต (lb, ub) และคำตอบจริง ใช้ตอนทดลองและวาดกราฟ
TEST_FUNCTIONS = {
    "Rosenbrock": {"func": rosenbrock, "lb": [-5.0, -5.0], "ub": [5.0, 5.0],
                   "x_star": [1.0, 1.0], "f_star": 0.0},
    "Rastrigin": {"func": rastrigin, "lb": [-5.12, -5.12], "ub": [5.12, 5.12],
                  "x_star": [0.0, 0.0], "f_star": 0.0},
}


class CountedFunction:
    # ห่อ objective function ไว้ เพื่อนับจำนวนครั้งที่ถูกเรียก (NFE)
    def __init__(self, func):
        self.func = func
        self.nfe = 0

    def __call__(self, x):
        self.nfe += 1          # เรียก f หนึ่งครั้ง = NFE เพิ่ม 1
        return self.func(x)


for name, info in TEST_FUNCTIONS.items():
    print(f"{name:10} f(x*) = f({info['x_star']}) = {info['func'](np.array(info['x_star'])):g}")

## 2. ตัวอัลกอริทึม Firefly

### สมการที่ใช้ (เลขสมการตรงกับ comment ในโค้ด)

| # | สมการ | ความหมาย |
|:-:|---|---|
| (1) | $I_i = -f(\mathbf{x}_i)$ | ความสว่าง — ยิ่ง $f$ ต่ำ ยิ่งสว่าง |
| (2) | $r_{ij}^2 = \lVert \mathbf{x}_i - \mathbf{x}_j \rVert^2 = \sum_k (x_{j,k} - x_{i,k})^2$ | ระยะห่างกำลังสองระหว่างหิ่งห้อย $i$ กับ $j$ |
| (3) | $\beta_{ij} = \beta_0 \, e^{-\gamma r_{ij}^2}$ | ความดึงดูด ลดลงตามระยะทาง |
| (4) | $\boldsymbol{\varepsilon} = (\text{rand} - 0.5) \times (\mathbf{ub} - \mathbf{lb})$ | เวกเตอร์สุ่ม, rand $\sim U(0,1)$ แยกแต่ละมิติ |
| (5) | $\mathbf{x}_i \leftarrow \mathbf{x}_i + \beta_{ij}(\mathbf{x}_j - \mathbf{x}_i) + \alpha_t \boldsymbol{\varepsilon}$ | **สมการเคลื่อนที่** เมื่อ $I_j > I_i$ |
| (6) | $\mathbf{x}_{\text{best}} \leftarrow \mathbf{x}_{\text{best}} + \alpha_t \boldsymbol{\varepsilon}$ | หิ่งห้อยที่สว่างที่สุดเดินสุ่มอย่างเดียว |
| (7) | $\alpha_t = \alpha_0 \, \theta^{t}$ | ลดการสุ่มลงทุกรอบ ($t = 0, 1, \dots$) |
| (8) | $\mathbf{x}_i \leftarrow \min(\max(\mathbf{x}_i, \mathbf{lb}), \mathbf{ub})$ | clip กลับเข้าขอบเขต |

### ลำดับการทำงานในแต่ละรอบ $t$

1. คำนวณ $\alpha_t$ ตามสมการ (7)
2. สำหรับหิ่งห้อย $i = 1 \dots n$: เทียบกับ $j = 1 \dots n$ ทีละคู่ — ถ้า $I_j > I_i$ ขยับ $i$ ตามสมการ (5) แล้ว **ประเมิน $f$ ใหม่ทันที** (sequential: คู่ถัดไปใช้ตำแหน่ง/ความสว่างล่าสุด)
3. ถ้าหิ่งห้อย $i$ ไม่เจอตัวที่สว่างกว่าเลย = มันสว่างที่สุด ณ ตอนนั้น → เดินสุ่มตามสมการ (6)
4. บันทึก **best-so-far** (คำตอบดีที่สุดเท่าที่เคยพบ) — ใช้วาด convergence curve

> **หมายเหตุ:** ตัวที่สว่างที่สุดเดินสุ่มแล้วอาจแย่ลงได้ แต่เราเก็บ best-so-far แยกไว้ จึงไม่สูญเสียคำตอบที่ดีที่สุดที่เคยเจอ

In [ ]:
def _fmt(v):
    # แสดงตัวเลข/เวกเตอร์เป็นทศนิยม 4 ตำแหน่ง (ใช้ในโหมดคำนวณด้วยมือ)
    if np.ndim(v) == 0:
        return f"{float(v):.4f}"
    return "(" + ", ".join(f"{float(a):.4f}" for a in v) + ")"


def _print_swarm(X, F, title):
    print(title)
    print(f"   {'firefly':>8} | {'position':>20} | {'f(x)':>12} | {'I = -f':>12}")
    for k, (x, fx) in enumerate(zip(X, F), start=1):
        print(f"   {k:>8} | {_fmt(x):>20} | {fx:>12.4f} | {-fx:>12.4f}")
    b = int(np.argmin(F))
    print(f"   -> สว่างที่สุด: หิ่งห้อย {b + 1}  (f = {F[b]:.4f})\n")


def firefly_algorithm(func, lb, ub, n_fireflies=N_FIREFLIES, max_iter=MAX_ITER, beta0=BETA0,
                      gamma=GAMMA, alpha0=ALPHA0, theta=THETA, seed=SEED, init_positions=None,
                      record_history=False, verbose=False, decimals=None):
    # decimals: ถ้ากำหนด (เช่น 4) จะปัดทุกค่ากลางทางก่อนใช้ต่อ -> ใช้ในโหมดคำนวณด้วยมือ
    #           เพื่อให้ตัวเลขบนสไลด์คำนวณตามด้วยเครื่องคิดเลขได้ตรงทุกหลัก
    rnd = (lambda v: np.round(v, decimals)) if decimals is not None else (lambda v: v)

    rng = np.random.default_rng(seed)             # ตัวสุ่มที่กำหนด seed ได้ -> ผลทำซ้ำได้
    lb = np.asarray(lb, dtype=float)
    ub = np.asarray(ub, dtype=float)
    dim = lb.size
    width = ub - lb                               # ความกว้างของขอบเขตแต่ละมิติ (ใช้ในสมการ 4)
    f = CountedFunction(func)                     # ตัวนับ NFE
    t_start = time.perf_counter()

    # ---------- ขั้นเริ่มต้น: วางหิ่งห้อยแบบสุ่มในขอบเขต (หรือใช้ตำแหน่งที่กำหนดเอง) ----------
    if init_positions is None:
        X = rng.uniform(lb, ub, size=(n_fireflies, dim))
    else:
        X = np.array(init_positions, dtype=float)
        n_fireflies = len(X)
    X = rnd(X)
    F = np.array([rnd(f(x)) for x in X])          # ค่า f ของหิ่งห้อยทุกตัว (ความสว่าง I = -F)

    best_idx = int(np.argmin(F))
    best_x, best_f = X[best_idx].copy(), float(F[best_idx])   # best-so-far
    curve = [best_f]                              # convergence curve: best-so-far หลังแต่ละรอบ
    history = [X.copy()] if record_history else None
    if verbose:
        _print_swarm(X, F, "=== ตำแหน่งเริ่มต้น (iteration 0) ===")

    for t in range(max_iter):
        alpha = rnd(alpha0 * theta ** t)          # (7) α_t = α0 · θ^t
        if verbose:
            print("=" * 78)
            print(f"ITERATION {t + 1}:  alpha_t = alpha0 * theta^{t} = {alpha0} * {theta}^{t} = {_fmt(alpha)}")
            print("=" * 78)

        for i in range(n_fireflies):
            moved = False
            for j in range(n_fireflies):
                if i == j:
                    continue
                I_i, I_j = -F[i], -F[j]           # (1) ความสว่าง I = -f
                if verbose:
                    print(f"[i={i + 1}, j={j + 1}]  I_{j + 1} = {_fmt(I_j)}  >  I_{i + 1} = {_fmt(I_i)} ?  "
                          + ("ใช่ -> หิ่งห้อย %d บินเข้าหา %d" % (i + 1, j + 1) if I_j > I_i else "ไม่ -> ข้าม"))
                if I_j > I_i:
                    diff = rnd(X[j] - X[i])                       # (x_j - x_i)
                    r2 = rnd(np.sum(diff ** 2))                   # (2) r² = Σ (x_j - x_i)²
                    beta = rnd(beta0 * np.exp(-gamma * r2))       # (3) β = β0 · exp(-γ r²)
                    rand = rnd(rng.random(dim))                   # rand ~ U(0, 1) แต่ละมิติ
                    eps = rnd((rand - 0.5) * width)               # (4) ε = (rand - 0.5) × ความกว้างขอบเขต
                    attract = rnd(beta * diff)                    # เทอมดึงดูด β (x_j - x_i)
                    random_step = rnd(alpha * eps)                # เทอมสุ่ม α ε
                    x_new = rnd(X[i] + attract + random_step)     # (5) สมการเคลื่อนที่
                    x_new = np.clip(x_new, lb, ub)                # (8) clip กลับเข้าขอบเขต
                    f_new = rnd(f(x_new))                         # ประเมิน f ใหม่ทันที (sequential)
                    if verbose:
                        print(f"      x_i = {_fmt(X[i])}   x_j = {_fmt(X[j])}")
                        print(f"      x_j - x_i = {_fmt(diff)}")
                        print(f"      r^2  = {' + '.join(f'({d:.4f})^2' for d in diff)} = {_fmt(r2)}")
                        print(f"      beta = {beta0} * exp(-{gamma} * {_fmt(r2)}) = {_fmt(beta)}")
                        print(f"      beta*(x_j - x_i) = {_fmt(attract)}")
                        print(f"      rand = {_fmt(rand)}  ->  eps = (rand - 0.5) * {_fmt(width)} = {_fmt(eps)}")
                        print(f"      alpha*eps = {_fmt(alpha)} * {_fmt(eps)} = {_fmt(random_step)}")
                        print(f"      x_new = {_fmt(X[i])} + {_fmt(attract)} + {_fmt(random_step)} = {_fmt(x_new)}")
                        print(f"      f_new = {_fmt(f_new)}   (เดิม {_fmt(F[i])})\n")
                    X[i], F[i] = x_new, f_new
                    moved = True

            if not moved:
                # (6) ไม่มีใครสว่างกว่า = ตัวที่สว่างที่สุด -> เดินสุ่มด้วยพจน์ α ε อย่างเดียว
                rand = rnd(rng.random(dim))
                eps = rnd((rand - 0.5) * width)                   # (4)
                random_step = rnd(alpha * eps)
                x_new = np.clip(rnd(X[i] + random_step), lb, ub)  # (6) + (8)
                f_new = rnd(f(x_new))
                if verbose:
                    print(f"   หิ่งห้อย {i + 1} ไม่มีตัวที่สว่างกว่า -> สว่างที่สุด -> เดินสุ่ม (สมการ 6)")
                    print(f"      rand = {_fmt(rand)}  ->  eps = {_fmt(eps)}")
                    print(f"      alpha*eps = {_fmt(random_step)}")
                    print(f"      x_new = {_fmt(X[i])} + {_fmt(random_step)} = {_fmt(x_new)}")
                    print(f"      f_new = {_fmt(f_new)}   (เดิม {_fmt(F[i])})\n")
                X[i], F[i] = x_new, f_new

        # ---------- ปรับปรุง best-so-far หลังจบรอบ ----------
        k = int(np.argmin(F))
        if F[k] < best_f:
            best_x, best_f = X[k].copy(), float(F[k])
        curve.append(best_f)
        if record_history:
            history.append(X.copy())
        if verbose:
            _print_swarm(X, F, f"--- สรุปหลัง iteration {t + 1} ---   best-so-far f = {best_f:.4f} ที่ {_fmt(best_x)}")

    return {
        "best_x": best_x,
        "best_f": best_f,
        "curve": np.array(curve),                  # ยาว max_iter + 1 (รวมรอบที่ 0)
        "nfe": f.nfe,
        "runtime": time.perf_counter() - t_start,
        "history": history,
    }

## 3. โหมดตรวจการคำนวณด้วยมือ (Hand Calculation Mode)

- หิ่งห้อย **3 ตัว** ตำแหน่งเริ่มต้นกำหนดเอง, **seed คงที่**, รัน **2 iterations** บน **Rosenbrock**
- พิมพ์ทุกขั้น: ตำแหน่ง, $f$, ความสว่าง, คู่ที่เปรียบเทียบ, $r^2$, $\beta$, $(\mathbf{x}_j - \mathbf{x}_i)$, rand, $\boldsymbol\varepsilon$, $\alpha\boldsymbol\varepsilon$, ตำแหน่งใหม่ และ $f$ ใหม่
- **ทุกค่าถูกปัดเป็นทศนิยม 4 ตำแหน่งก่อนนำไปใช้ขั้นถัดไป** → เอาตัวเลขที่พิมพ์ออกมาไปคำนวณต่อด้วยเครื่องคิดเลขได้ตรงทุกหลัก (เหมาะทำสไลด์)
- ใช้ฟังก์ชัน `firefly_algorithm()` **ตัวเดียวกับที่ใช้ทดลองจริง** แค่เปิด `verbose=True, decimals=4` → พิสูจน์ได้ว่าสไลด์ตรงกับโค้ดจริง

ตำแหน่งเริ่มต้นที่เลือก: $A = (0, 0)$ มี $f = 1$ · $B = (-1, 0.5)$ มี $f = 4 + 25 = 29$ · $C = (1.5, 2)$ มี $f = 0.25 + 6.25 = 6.5$

In [ ]:
def hand_calculation_demo(init_positions=((0.0, 0.0), (-1.0, 0.5), (1.5, 2.0)), seed=7, n_iter=2):
    info = TEST_FUNCTIONS["Rosenbrock"]
    print("HAND CALCULATION MODE — Firefly Algorithm บน Rosenbrock")
    print(f"พารามิเตอร์: beta0 = {BETA0}, gamma = {GAMMA}, alpha0 = {ALPHA0}, theta = {THETA}, seed = {seed}")
    print(f"ขอบเขต: lb = {info['lb']}, ub = {info['ub']}  ->  ความกว้าง = {np.subtract(info['ub'], info['lb'])}")
    print("สูตร: x_i = x_i + beta0*exp(-gamma*r^2)*(x_j - x_i) + alpha*eps,  eps = (rand - 0.5)*(ub - lb)")
    print("ทุกค่าปัดเป็นทศนิยม 4 ตำแหน่งก่อนใช้ในขั้นถัดไป\n")
    result = firefly_algorithm(info["func"], info["lb"], info["ub"], max_iter=n_iter, seed=seed,
                               init_positions=init_positions, verbose=True, decimals=4)
    print(f"ผลลัพธ์หลัง {n_iter} iterations: best-so-far x = {_fmt(result['best_x'])}, "
          f"f = {result['best_f']:.4f},  NFE = {result['nfe']}")
    return result


hand_result = hand_calculation_demo()

**วิธีนำไปทำสไลด์:** เลือกคู่ที่ "ใช่ → บินเข้าหา" มา 1–2 คู่ แล้วเขียนตามลำดับ
$\mathbf{x}_j - \mathbf{x}_i \;\to\; r^2 \;\to\; \beta \;\to\; \beta(\mathbf{x}_j - \mathbf{x}_i) \;\to\; \boldsymbol\varepsilon \;\to\; \alpha\boldsymbol\varepsilon \;\to\; \mathbf{x}_{\text{new}} \;\to\; f_{\text{new}}$
ทุกตัวเลขมีอยู่ใน output ด้านบนครบแล้ว

**ข้อสังเกตที่ควรพูดตอนนำเสนอ:**
- คู่ที่ $I_j \le I_i$ จะถูกข้าม — หิ่งห้อยบินเข้าหา **เฉพาะตัวที่สว่างกว่า**
- เมื่อ $r^2$ ใหญ่ $\beta$ จะเล็กมาก (แทบไม่ถูกดึง) — การเคลื่อนที่จึงมาจากเทอมสุ่ม $\alpha\boldsymbol\varepsilon$ เป็นหลัก
- ค่า $f$ ของหิ่งห้อยที่ขยับ **อาจแย่ลงได้** เพราะ FA ไม่ได้บังคับว่าต้องดีขึ้น (ต่างจาก greedy search) — แต่ best-so-far ไม่มีวันแย่ลง

## 4. การทดลอง: รันอิสระ `N_RUNS` ครั้งต่อฟังก์ชัน

แต่ละ run ใช้ seed ต่างกัน (`SEED + k`) จึงเป็นการรันที่เป็นอิสระต่อกันและทำซ้ำผลได้
เก็บ: fitness สุดท้าย (best-so-far), เวลาที่ใช้, NFE, convergence curve และตำแหน่งหิ่งห้อยทุกรอบของ run แรก (ใช้วาด contour)

In [ ]:
def run_experiments(name, n_runs=N_RUNS):
    info = TEST_FUNCTIONS[name]
    runs = []
    for k in range(n_runs):
        res = firefly_algorithm(info["func"], info["lb"], info["ub"], seed=SEED + k,
                                record_history=(k == 0))           # เก็บ history เฉพาะ run แรก
        runs.append(res)
    best_f = np.array([r["best_f"] for r in runs])
    return {
        "name": name,
        "runs": runs,
        "best_f": best_f,
        "runtime": np.array([r["runtime"] for r in runs]),
        "nfe": np.array([r["nfe"] for r in runs]),
        "curves": np.stack([r["curve"] for r in runs]),         # รูปร่าง (n_runs, max_iter + 1)
        "success_rate": np.mean(best_f < SUCCESS_TOL),
        "best_run": runs[int(np.argmin(best_f))],
    }


def print_summary_table(results):
    header = (f"{'Function':<12}{'Best':>12}{'Mean':>12}{'Worst':>12}{'Std':>12}"
              f"{'Avg time(s)':>13}{'Avg NFE':>10}{'Success':>10}")
    print(header)
    print("-" * len(header))
    for R in results:
        f = R["best_f"]
        print(f"{R['name']:<12}{f.min():>12.3e}{f.mean():>12.3e}{f.max():>12.3e}{f.std():>12.3e}"
              f"{R['runtime'].mean():>13.4f}{R['nfe'].mean():>10.0f}{R['success_rate']:>10.0%}")
    print(f"\n(ผลจาก {N_RUNS} runs ต่อฟังก์ชัน, Success = สัดส่วนที่ได้ f < {SUCCESS_TOL:g})")
    for R in results:
        b = R["best_run"]
        print(f"{R['name']:<12} คำตอบดีที่สุด x = {np.round(b['best_x'], 6)},  f = {b['best_f']:.3e}")


def analyze_runs(R, valley_radius=0.5):
    # แยก run ออกเป็น 3 กลุ่ม เพื่อดูว่า "ไม่สำเร็จ" เพราะอะไร
    info = TEST_FUNCTIONS[R["name"]]
    dist = np.array([np.linalg.norm(r["best_x"] - np.array(info["x_star"])) for r in R["runs"]])
    ok = R["best_f"] < SUCCESS_TOL
    near = (~ok) & (dist < valley_radius)          # อยู่หุบที่ถูกแล้ว แต่ยังไม่แม่นพอ
    other = (~ok) & (dist >= valley_radius)        # ติดหุบอื่น (local minimum)
    print(f"{R['name']:<12} สำเร็จ (f < {SUCCESS_TOL:g}): {ok.sum():>2}/{len(ok)}   |   "
          f"อยู่หุบที่ถูกแต่ยังไม่แม่นพอ: {near.sum():>2}   |   ติดหุบอื่น (local min): {other.sum():>2}")
    if other.any():
        where = [f"({x[0]:.2f}, {x[1]:.2f})" for x, o in zip([r["best_x"] for r in R["runs"]], other) if o]
        print(f"{'':<12} ตำแหน่งที่ติด: {', '.join(where)}")


results = [run_experiments(name) for name in TEST_FUNCTIONS]
print_summary_table(results)
print()
for R in results:
    analyze_runs(R)

## 5. กราฟผลลัพธ์ (บันทึกเป็น .png ทุกภาพ)

### 5.1 Convergence Curve
เส้นทึบ = ค่าเฉลี่ยของ best-so-far จากทุก run · แถบสี = ช่วง min–max ของทุก run · แกน y เป็น log

In [ ]:
def plot_convergence(R, color):
    curves = np.maximum(R["curves"], 1e-16)          # กัน log(0)
    iters = np.arange(curves.shape[1])
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.fill_between(iters, curves.min(axis=0), curves.max(axis=0), color=color, alpha=0.25,
                    label=f"min-max over {len(curves)} runs")
    ax.plot(iters, curves.mean(axis=0), color=color, lw=2.2, label="mean best-so-far")
    ax.set_yscale("log")
    ax.set(xlabel="iteration", ylabel="best-so-far f (log scale)",
           title=f"Firefly Algorithm convergence: {R['name']}")
    ax.grid(alpha=0.3, which="both")
    ax.legend()
    fig.tight_layout()
    path = os.path.join(FIG_DIR, f"convergence_{R['name'].lower()}.png")
    fig.savefig(path, dpi=150)
    plt.show()
    print("saved:", path)


for R, color in zip(results, ["tab:blue", "tab:red"]):
    plot_convergence(R, color)

### 5.2 ตำแหน่งหิ่งห้อยบน contour ที่ iteration 0, 5, 20 และสุดท้าย (จาก run แรก)

🟠 = หิ่งห้อย · 🔷 = ตัวที่สว่างที่สุดในรอบนั้น · ⭕ วงแดง = คำตอบจริง
Rosenbrock ใช้ระดับ contour แบบ log เพราะค่า $f$ ไล่ตั้งแต่ 0 ถึงหลายหมื่น

> **อ่านภาพรอบสุดท้าย:** หิ่งห้อยส่วนใหญ่มักไปซ้อนกันที่จุดเดียว จึงเห็นจุดน้อยกว่า 25 จุด — ข้อความใต้ภาพจะบอกจำนวนที่ซ้อนกัน และบอกว่า run นี้สำเร็จหรือติดหุบข้างเคียง

In [ ]:
def plot_swarm_snapshots(R, snapshot_iters=SNAPSHOT_ITERS):
    info = TEST_FUNCTIONS[R["name"]]
    history = R["runs"][0]["history"]
    lb, ub = info["lb"], info["ub"]
    gx = np.linspace(lb[0], ub[0], 300)
    gy = np.linspace(lb[1], ub[1], 300)
    GX, GY = np.meshgrid(gx, gy)
    GZ = info["func"]((GX, GY))
    if R["name"] == "Rosenbrock":
        levels = np.logspace(-1, np.log10(GZ.max()), 25)
    else:
        levels = 15

    fig, axes = plt.subplots(1, len(snapshot_iters), figsize=(4.4 * len(snapshot_iters), 4.6))
    for ax, t in zip(axes, snapshot_iters):
        P = history[t]
        vals = np.array([info["func"](p) for p in P])
        ax.contour(GX, GY, GZ, levels=levels, cmap="viridis", linewidths=0.7)
        ax.scatter(P[:, 0], P[:, 1], s=30, c="tab:orange", edgecolors="k", linewidths=0.5, zorder=3)
        ax.plot(*P[vals.argmin()], "D", ms=8, mfc="deepskyblue", mec="k", zorder=4)
        ax.plot(*info["x_star"], "o", ms=15, mfc="none", mec="red", mew=2, zorder=2)
        ax.set(xlim=(lb[0], ub[0]), ylim=(lb[1], ub[1]), xlabel="x", ylabel="y",
               title=f"iteration {t}\nbest f = {vals.min():.2e}")
        ax.set_aspect("equal")
    fig.suptitle(f"{R['name']} (run 1): firefly positions (orange), brightest (blue), true minimum (red ring)")
    fig.tight_layout(rect=(0, 0, 1, 0.93))
    path = os.path.join(FIG_DIR, f"swarm_{R['name'].lower()}.png")
    fig.savefig(path, dpi=150)
    plt.show()

    # บอกสถานะของ run ที่นำมาวาด เพื่อให้อ่านภาพถูก
    run = R["runs"][0]
    final = history[-1]
    center = final[np.argmin([info["func"](p) for p in final])]
    stacked = int(np.sum(np.linalg.norm(final - center, axis=1) < 0.05))
    status = "สำเร็จ" if run["best_f"] < SUCCESS_TOL else "ไม่สำเร็จ"
    print(f"run 1: best-so-far f = {run['best_f']:.3e} ที่ {np.round(run['best_x'], 4)} -> {status}")
    print(f"       ตัวที่สว่างที่สุดในรอบสุดท้าย อยู่ที่ {np.round(center, 4)}  f = {info['func'](center):.3e}")
    if np.linalg.norm(center - run["best_x"]) > 0.5:
        print("       !! ฝูงรอบสุดท้ายอยู่คนละที่กับ best-so-far: ตัวที่สว่างที่สุดเดินสุ่ม (สมการ 6) ออกจากจุดดี "
              "แล้วกลับมาไม่ได้ -> นี่คือเหตุผลที่ต้องเก็บ best-so-far แยกไว้")
    print(f"       รอบสุดท้าย มีหิ่งห้อย {stacked}/{len(final)} ตัวซ้อนกันอยู่รอบตัวที่สว่างที่สุด "
          f"(จุดซ้อนกันจึงเห็นน้อยกว่าจำนวนจริง)")
    print("saved:", path)


for R in results:
    plot_swarm_snapshots(R)

### 5.3 อ่านผลการทดลอง

- **Rosenbrock:** ด้วยพารามิเตอร์ตั้งต้น สำเร็จทุก run — หุบเขารูปกล้วยมีหุบเดียว ฝูงหิ่งห้อยค่อยๆ ไหลไปตามหุบจนถึง $(1, 1)$
- **Rastrigin:** อัตราสำเร็จต่ำกว่ามาก แต่ดูผลวิเคราะห์ใต้ตารางจะเห็นว่า run ที่ "ไม่สำเร็จ" มี **2 แบบ**:
  1. **อยู่หุบที่ถูกแล้วแต่ยังไม่แม่นพอ** (เป็นส่วนใหญ่) — หลุมของ Rastrigin ชันมาก คลาดจาก $(0,0)$ ไปแค่ 0.01 ก็ได้ $f \approx 0.02$ แล้ว ขณะที่ในรอบท้ายๆ $\alpha$ ยังทำให้กระโดดทีละประมาณ $\pm 0.05$ จึงลงไม่ถึงก้นหลุมระดับ $10^{-4}$
  2. **ติดหุบข้างเคียง** เช่น $(\pm 1, 0)$ หรือ $(0, \pm 1)$ ซึ่ง $f \approx 1$ — ต้องข้าม "สันเขา" ที่ $f$ สูงกว่าจึงจะย้ายหุบได้ และเมื่อ $\alpha$ ลดลงการสุ่มก็เล็กเกินกว่าจะข้าม
  → FA **หาหุบที่ถูกได้ดี** (exploration) แต่ **เก็บรายละเอียดช่วงท้ายได้ไม่แม่น** (exploitation) — แก้ได้ด้วยการลด $\alpha$ ให้เร็วขึ้นช่วงท้าย หรือเพิ่มรอบ
- **ภาพ Rastrigin (run 1):** best-so-far อยู่ในหุบที่ถูกใกล้ $(0, 0)$ แต่ฝูงรอบสุดท้ายกลับไปรวมกันที่หุบข้างเคียง $(-1, 0)$ —
  เพราะ FA แบบมาตรฐาน **ไม่มี elitism**: ตัวที่สว่างที่สุดต้องเดินสุ่มตามสมการ (6) แม้จะทำให้แย่ลง เมื่อมันหลุดจากจุดดี ฝูงก็ตามมันไป
  การเก็บ best-so-far แยกไว้ในโค้ดจึงจำเป็น (เป็นประเด็นที่ดีสำหรับตอบคำถามตอนนำเสนอ)
- **NFE ≈ 29,000 ต่อ run** ทั้งสองฟังก์ชัน: เริ่มต้น $n = 25$ ครั้ง + ทุกครั้งที่หิ่งห้อยขยับ (ประมาณ $n^2/2 \approx 300$ ครั้งต่อรอบ × 100 รอบ)
  → ต้นทุนของ FA เป็น $O(n^2)$ ต่อรอบ ต่างจาก PSO ที่เป็น $O(n)$

**ลองปรับพารามิเตอร์ (ส่วนที่ 0):** เพิ่ม `N_FIREFLIES`, เพิ่ม `MAX_ITER` หรือปรับ `THETA` ให้ α ลดช้าลง แล้วดูว่าอัตราสำเร็จของ Rastrigin เปลี่ยนอย่างไร

## 6. ตัวอย่างปัญหาจริง: Tension/Compression Spring Design

ออกแบบ **สปริงขด** ให้มี **น้ำหนักน้อยที่สุด** ภายใต้เงื่อนไขทางวิศวกรรม — เป็นปัญหามาตรฐานที่ใช้เปรียบเทียบอัลกอริทึม optimization ในงานวิจัย (เริ่มจาก Arora, 1989)

### ตัวแปรออกแบบ (3 ตัว)

| ตัวแปร | ความหมาย | ขอบเขต |
|---|---|---|
| $x_1 = d$ | เส้นผ่านศูนย์กลางของ **ลวด** (wire diameter) | $0.05 \le d \le 2.0$ |
| $x_2 = D$ | เส้นผ่านศูนย์กลางเฉลี่ยของ **ขด** (mean coil diameter) | $0.25 \le D \le 1.3$ |
| $x_3 = N$ | **จำนวนขด** ที่ทำงาน (active coils) | $2 \le N \le 15$ |

### ฟังก์ชันเป้าหมาย — น้ำหนักสปริง (สัดส่วนกับปริมาตรลวด)
$$ \min\; f(\mathbf{x}) = (N + 2)\, D\, d^2 $$

### เงื่อนไขบังคับ 4 ข้อ ($g_k \le 0$)

| ข้อ | สมการ | ความหมายทางวิศวกรรม |
|---|---|---|
| $g_1$ | $1 - \dfrac{D^3 N}{71785\, d^4} \le 0$ | **ระยะยุบตัวขั้นต่ำ** (minimum deflection) — สปริงต้องยุบได้พอภายใต้แรงที่กำหนด |
| $g_2$ | $\dfrac{4D^2 - dD}{12566(D d^3 - d^4)} + \dfrac{1}{5108\, d^2} - 1 \le 0$ | **ความเค้นเฉือน** (shear stress) ในลวดต้องไม่เกินค่าที่ยอมรับได้ |
| $g_3$ | $1 - \dfrac{140.45\, d}{D^2 N} \le 0$ | **ความถี่ธรรมชาติ** (surge frequency) ต้องสูงพอ ไม่เกิดการสั่นพ้อง |
| $g_4$ | $\dfrac{D + d}{1.5} - 1 \le 0$ | **เส้นผ่านศูนย์กลางภายนอก** $D + d$ ต้องไม่เกิน 1.5 (ใส่ในช่องที่กำหนดได้) |

### จัดการเงื่อนไขบังคับด้วย Penalty Function

FA ถูกออกแบบมาสำหรับปัญหาที่ไม่มีเงื่อนไข จึงแปลงเป็นปัญหาไม่มีเงื่อนไขโดย **บวกค่าปรับ** เมื่อฝ่าฝืน:
$$ F(\mathbf{x}) = f(\mathbf{x}) + P \sum_{k=1}^{4} \big[\max(0,\, g_k(\mathbf{x}))\big]^2 , \qquad P = 10^6 $$
- ถ้าทำตามเงื่อนไขครบ ($g_k \le 0$ ทุกข้อ) → ค่าปรับเป็น 0, $F = f$
- ถ้าฝ่าฝืน → ค่าปรับโตตามกำลังสองของการฝ่าฝืน ทำให้หิ่งห้อยตัวนั้น "มืด" และถูกดึงไปหาตัวที่ทำตามเงื่อนไข

> **หมายเหตุ:** ในโลกจริง $N$ ต้องเป็นจำนวนเต็ม แต่ในโจทย์มาตรฐานนี้ถือเป็นตัวแปรต่อเนื่อง เพื่อให้เทียบกับงานวิจัยได้

In [ ]:
SPRING_LB = [0.05, 0.25, 2.0]        # ขอบเขตล่างของ (d, D, N)
SPRING_UB = [2.0, 1.3, 15.0]         # ขอบเขตบนของ (d, D, N)
PENALTY = 1e6                        # P : ค่าสัมประสิทธิ์ของค่าปรับ
SPRING_N_RUNS = 10
SPRING_MAX_ITER = 300                # ปัญหา 3 มิติที่มีเงื่อนไข ต้องใช้รอบมากกว่าฟังก์ชันทดสอบ


def spring_weight(x):
    # f(x) = (N + 2) · D · d²   (น้ำหนักสปริง)
    d, D, N = x
    return (N + 2) * D * d ** 2


def spring_constraints(x):
    # คืนค่า [g1, g2, g3, g4] — ต้อง <= 0 ทุกตัวจึงจะเป็นคำตอบที่ใช้ได้ (feasible)
    d, D, N = x
    g1 = 1 - (D ** 3 * N) / (71785 * d ** 4)                                              # ระยะยุบตัวขั้นต่ำ
    g2 = (4 * D ** 2 - d * D) / (12566 * (D * d ** 3 - d ** 4)) + 1 / (5108 * d ** 2) - 1  # ความเค้นเฉือน
    g3 = 1 - 140.45 * d / (D ** 2 * N)                                                    # ความถี่ธรรมชาติ
    g4 = (D + d) / 1.5 - 1                                                                # เส้นผ่านศูนย์กลางภายนอก
    return np.array([g1, g2, g3, g4])


def spring_penalized(x):
    # F(x) = f(x) + P · Σ max(0, g_k)²   -> ฟังก์ชันที่ FA ใช้หาค่าต่ำสุดจริงๆ
    violation = np.maximum(0.0, spring_constraints(x))
    return spring_weight(x) + PENALTY * np.sum(violation ** 2)


# ค่าอ้างอิงจากงานวิจัย (x ตามที่ตีพิมพ์ แล้วคำนวณ f และ g ใหม่ด้วยโค้ดเดียวกัน)
SPRING_REFERENCES = {
    "Arora (1989)": [0.053396, 0.399180, 9.185400],
    "Coello (2000)": [0.051480, 0.351661, 11.632201],
    "Best known (literature)": [0.051690, 0.356750, 11.287126],
}


def real_world_demo(n_runs=SPRING_N_RUNS, max_iter=SPRING_MAX_ITER):
    runs = [firefly_algorithm(spring_penalized, SPRING_LB, SPRING_UB, max_iter=max_iter, seed=SEED + k)
            for k in range(n_runs)]
    weights = np.array([spring_weight(r["best_x"]) for r in runs])
    feasible = np.array([np.all(spring_constraints(r["best_x"]) <= 1e-6) for r in runs])
    best = runs[int(np.argmin([r["best_f"] for r in runs]))]
    x = best["best_x"]

    print(f"Tension/Compression Spring Design — FA {n_runs} runs, n = {N_FIREFLIES}, max_iter = {max_iter}")
    print(f"น้ำหนัก f: best = {weights.min():.6f}, mean = {weights.mean():.6f}, "
          f"worst = {weights.max():.6f}, std = {weights.std():.2e}")
    print(f"feasible runs: {feasible.sum()}/{n_runs},   avg NFE = {np.mean([r['nfe'] for r in runs]):.0f},   "
          f"avg time = {np.mean([r['runtime'] for r in runs]):.2f} s\n")

    rows = [("FA (this work, best run)", x)] + [(k, np.array(v)) for k, v in SPRING_REFERENCES.items()]
    print(f"{'Source':<26}{'d':>10}{'D':>10}{'N':>10}{'f(x)':>12}{'g1':>11}{'g2':>11}{'g3':>9}{'g4':>9}")
    print("-" * 108)
    for label, xv in rows:
        g = spring_constraints(xv)
        print(f"{label:<26}{xv[0]:>10.6f}{xv[1]:>10.6f}{xv[2]:>10.6f}{spring_weight(xv):>12.7f}"
              + "".join(f"{v:>11.2e}" for v in g[:2]) + "".join(f"{v:>9.4f}" for v in g[2:]))
    gap = (spring_weight(x) - spring_weight(np.array(SPRING_REFERENCES["Best known (literature)"])))
    print(f"\nFA ห่างจากค่าที่ดีที่สุดที่รู้จัก: {gap:.2e} ({gap / 0.012665 * 100:.3f}%)")

    print(f"\nคำตอบของแต่ละ run  (* = ได้น้ำหนัก < 0.0127 คือใกล้ค่าที่ดีที่สุด)")
    print(f"{'run':>4}{'d':>10}{'D':>10}{'N':>10}{'f(x)':>12}")
    for k, (r, w) in enumerate(zip(runs, weights), start=1):
        d, D, N = r["best_x"]
        print(f"{k:>4}{d:>10.5f}{D:>10.5f}{N:>10.4f}{w:>12.7f} {'*' if w < 0.0127 else ''}")

    curves = np.stack([r["curve"] for r in runs])
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    it = np.arange(curves.shape[1])
    axes[0].fill_between(it, curves.min(0), curves.max(0), color="tab:green", alpha=0.25, label="min-max")
    axes[0].plot(it, curves.mean(0), color="tab:green", lw=2, label="mean best-so-far")
    axes[0].axhline(0.012665, color="k", ls="--", lw=1, label="best known 0.012665")
    axes[0].set_yscale("log")
    axes[0].set_ylim(0.0125, 0.03)       # ตัดแกน: รอบแรกๆ มีค่าปรับสูงถึง 1e5 จะบังช่วงที่น่าสนใจ
    axes[0].set(xlabel="iteration", ylabel="penalized weight F (log, zoomed)",
                title="Spring design: convergence (zoomed to 0.0125-0.03)")
    axes[0].legend(); axes[0].grid(alpha=0.3, which="both")
    axes[1].bar(range(1, n_runs + 1), weights, color=np.where(weights < 0.0127, "tab:green", "tab:gray"))
    axes[1].axhline(0.012665, color="k", ls="--", lw=1)
    axes[1].set_ylim(weights.min() * 0.995, weights.max() * 1.005)
    axes[1].set(xlabel="run", ylabel="final weight f", title="Final weight of each run (dashed = best known)")
    fig.tight_layout()
    path = os.path.join(FIG_DIR, "spring_design.png")
    fig.savefig(path, dpi=150)
    plt.show()
    print("saved:", path)
    return runs


spring_runs = real_world_demo()

**อ่านผล:**
- ทุกคำตอบของ FA ทำตามเงื่อนไขครบ (feasible) — ค่า $g_1, g_2$ ใกล้ 0 มาก แปลว่าคำตอบที่ดีที่สุดอยู่ **บนขอบของเงื่อนไข** (active constraints) ซึ่งเป็นเรื่องปกติของปัญหาออกแบบ: ยิ่งเบาเท่าไร ยิ่งเฉียดขีดจำกัดความแข็งแรง
- run ที่ไม่มี `*` ในตาราง (แท่งสีเทาในกราฟ) ติด **local optimum** — ปัญหานี้มีคำตอบ "พอใช้ได้" หลายแบบ เช่น
  ลวดเส้นเล็กสุด $d = 0.05$ (ติดขอบเขตล่าง) กับขดจำนวนมาก หรือ **ลวดหนากว่า ($d \approx 0.06$) กับขดน้อย ($N \approx 3$–$4$)** ซึ่งหนักกว่าคำตอบที่ดีที่สุดราว 3–20%
  → ในงานจริงจึงควร **รันหลายครั้งแล้วเลือกคำตอบที่ดีที่สุด** เหมือนที่ทำในส่วนนี้
- ค่า "Best known" ที่ตีพิมพ์มี $g_2$ เป็นบวกเล็กน้อย (~$10^{-5}$) เพราะตัวเลขถูกปัดเป็น 6 ตำแหน่งตอนตีพิมพ์ ไม่ได้แปลว่าคำตอบผิด

## 7. อธิบายโค้ดทีละส่วน เทียบกับสมการ FA (สำหรับนำเสนอ)

### ภาพรวม: ส่วนไหนในโค้ด = อะไรในทฤษฎี

| ในโค้ด (`firefly_algorithm`) | สมการ / ขั้นตอน FA | พูดตอนนำเสนอว่า |
|---|---|---|
| `X = rng.uniform(lb, ub, size=(n_fireflies, dim))` | การสร้างประชากรเริ่มต้น | "วางหิ่งห้อย $n$ ตัวแบบสุ่มกระจายทั่วพื้นที่ค้นหา" |
| `F = np.array([f(x) for x in X])` | ประเมิน $f(\mathbf{x}_i)$ | "วัดคุณภาพของแต่ละตำแหน่ง" — ทุกการเรียก `f` ถูกนับเป็น NFE |
| `I_i, I_j = -F[i], -F[j]` | **(1)** $I = -f$ | "ความสว่างคือค่าลบของ $f$ เพราะเราหาค่าต่ำสุด — $f$ ต่ำ = สว่าง" |
| `if I_j > I_i:` | กฎของหิ่งห้อย | "หิ่งห้อยบินเข้าหาเฉพาะตัวที่สว่างกว่า" |
| `diff = X[j] - X[i]` | $(\mathbf{x}_j - \mathbf{x}_i)$ | "ทิศทางจาก $i$ ไปหา $j$" |
| `r2 = np.sum(diff ** 2)` | **(2)** $r_{ij}^2 = \sum_k (x_{j,k} - x_{i,k})^2$ | "ระยะห่างกำลังสอง (Euclidean)" |
| `beta = beta0 * np.exp(-gamma * r2)` | **(3)** $\beta = \beta_0 e^{-\gamma r^2}$ | "ยิ่งไกลยิ่งดึงดูดน้อย แสงถูกอากาศดูดกลืน" |
| `rand = rng.random(dim)` | rand $\sim U(0,1)$ | "สุ่มตัวเลขแยกกันทุกมิติ" |
| `eps = (rand - 0.5) * width` | **(4)** $\boldsymbol\varepsilon = (\text{rand} - 0.5)(\mathbf{ub} - \mathbf{lb})$ | "เลขสุ่มในช่วง $[-0.5, 0.5]$ ขยายตามขนาดขอบเขต" |
| `x_new = X[i] + attract + random_step` | **(5)** $\mathbf{x}_i + \beta(\mathbf{x}_j - \mathbf{x}_i) + \alpha\boldsymbol\varepsilon$ | "เคลื่อนที่ = ถูกดึง (exploitation) + สุ่มสำรวจ (exploration)" |
| `f_new = f(x_new)` แล้วเขียนทับ `X[i], F[i]` | Sequential update | "อัปเดตทันที คู่ถัดไปเห็นตำแหน่งใหม่แล้ว" |
| `if not moved:` → `X[i] + random_step` | **(6)** ตัวที่สว่างที่สุดเดินสุ่ม | "ตัวที่ดีที่สุดไม่มีใครให้ตาม จึงสำรวจรอบๆ ตัวเอง" |
| `alpha = alpha0 * theta ** t` | **(7)** $\alpha_t = \alpha_0 \theta^t$ | "ช่วงแรกสุ่มเยอะเพื่อสำรวจ ช่วงหลังสุ่มน้อยเพื่อเก็บรายละเอียด" |
| `np.clip(x_new, lb, ub)` | **(8)** clip | "ถ้าบินหลุดขอบเขต ดึงกลับมาที่ขอบ" |
| `if F[k] < best_f:` + `curve.append(best_f)` | best-so-far | "จำคำตอบที่ดีที่สุดที่เคยเจอ — เส้น convergence จึงไม่มีวันขึ้น" |
| `CountedFunction.__call__` → `self.nfe += 1` | NFE | "ต้นทุนการคำนวณ วัดแบบไม่ขึ้นกับความเร็วเครื่อง" |

### ลำดับการเล่า (แนะนำ)

1. **ส่วนที่ 0 — พารามิเตอร์:** อธิบายบทบาทของ $\beta_0, \gamma, \alpha_0, \theta$ (ทุกตัวปรับได้ที่เดียว)
2. **ส่วนที่ 2 — ลูปหลัก:** วนรอบ $t$ → วน $i$ → วน $j$ → ถ้า $j$ สว่างกว่า ใช้สมการ (2)(3)(4)(5)(8) → ถ้าไม่มีใครสว่างกว่า ใช้สมการ (6)
3. **ส่วนที่ 3 — คำนวณด้วยมือ:** ชี้ให้เห็นว่าตัวเลขบนสไลด์ออกมาจากฟังก์ชันเดียวกับที่ใช้ทดลองจริง
4. **ส่วนที่ 4–5 — ผลการทดลอง:** Rosenbrock สำเร็จสูง vs Rastrigin ติด local minima → จุดแข็ง/จุดอ่อนของ FA
5. **ส่วนที่ 6 — ปัญหาจริง:** penalty function ทำให้ FA แก้ปัญหาที่มีเงื่อนไขได้ และได้ผลใกล้ค่าที่ดีที่สุดในงานวิจัย

### คำถามที่อาจถูกถาม
- **ทำไม $I = -f$?** เพราะ FA "ตามแสงที่สว่างกว่า" (หาค่ามาก) แต่เราต้องการค่าต่ำสุด จึงกลับเครื่องหมาย
- **$\gamma$ ส่งผลอย่างไร?** $\gamma \to 0$: ทุกตัวถูกดึงแรงเท่ากันหมด (คล้าย PSO) · $\gamma \to \infty$: แทบไม่มีแรงดึง (random search)
- **ทำไมต้องลด $\alpha$?** ถ้าไม่ลด ฝูงจะสั่นรอบคำตอบตลอด ไม่ลู่เข้า — ถ้าลดเร็วเกิน จะหยุดสำรวจเร็วเกินไปและติด local minimum
- **ทำไม NFE สูง?** ทุกคู่ $(i, j)$ ที่ $j$ สว่างกว่าต้องประเมิน $f$ ใหม่ → $O(n^2)$ ต่อรอบ